# 04 · Check alignment with known computation

**Question:** Do dimensionality, stage similarity, and lane values align with known
RASP computation, and do targeted interventions produce the predicted output changes?

These are hypotheses to test. PR need not equal a variable count, and a scheduled
write need not lower CKA. Correlation alone does not establish correct lane values.
This notebook runs independently and recreates [Step 3](03_geometry.ipynb)'s probes
with the same program order, seed, class balance, and target positions.

## Configure

Use all 14 programs for observational checks. For A/B, use all 640 target observations
with an unchanged-value control and one outcome-changing donor per target.

In [ ]:
import sys
from pathlib import Path

# Run from the repository root or any folder inside it.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (root for root in (start, *start.parents) if (root / "raspformer").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the RaspFormer repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

from raspformer.alignment import run_alignment_study
from raspformer.compiler import CompilerConfig
from raspformer.geometry import ProbeConfig
from raspformer.programs import build_programs

config = CompilerConfig(rel_tol=1e-4, abs_tol=1e-4)
probe_config = ProbeConfig()  # Step 3: seed 42, length 6, 64 targets per class.
programs = build_programs(config.max_seq_len, include_examples=True)
OUTPUT_ROOT = PROJECT_ROOT / "results" / "alignment"
VIEW = "computed"  # Change to "full" for all residual lanes.

## Run

Reuse the existing sampler, activation collector, PR/CKA functions, and manifest
writer. The numerical study saves tables and the exact probes before plotting.
A completed run means measurements were collected; it does not mean every hypothesis
or comparison passed. Review the mismatches below.

In [ ]:
result = run_alignment_study(programs, config, probe_config, OUTPUT_ROOT)
RUN_DIR = result["run_dir"]
FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir()

## Inspect · variable values and lanes

Evaluate each RASP intermediate on the same probes and encode it in the compiler's
basis: one-hot categorical values, scalar numerical values, and zeros for undefined
values. Compare lanes at their scheduled write and every later stage.

`fraction_close` uses the configured absolute and relative tolerance. Pearson
correlation is undefined for a constant expected or observed lane. It can be nearly
one even when lane magnitudes differ enough to fail the value comparison.

In [ ]:
lanes = result["lanes"]
lane_summary = lanes.groupby("program").agg(
    lane_stage_checks=("lane", "size"),
    max_abs_error=("max_abs_error", "max"),
    lowest_fraction_close=("fraction_close", "min"),
)
display(lane_summary)
mismatches = lanes[lanes.fraction_close < 1].sort_values("max_abs_error", ascending=False)
print(f"{len(mismatches)} lane/stage comparisons have at least one value outside tolerance.")
display(mismatches.head(20))

## Inspect · PR and adjacent-stage CKA

The expected representation contains **variables written so far**, in their compiler
encoding. Unwritten and compiler scratch lanes are zero. The full view also includes
all token, position, and constant-one embeddings. Thus, the PR gap can reflect scratch
signals or numerical approximation as well as the structure of the computation.
It is a comparison with this explicit benchmark, not a required zero-error metric.

PR uses globally centered observations without whitening. The plotted signed gap is
actual PR minus expected PR. CKA compares the same ordered observations at adjacent
stages; orange guides mark scheduled writes, including transitions with undefined CKA. Missing CKA values are undefined, not
zero. The embedding has no preceding stage. Constant written variables may add no
variance. No variable lifetime or live-variable count is inferred.

In [ ]:
stages = result["stages"]
selected = stages[stages.view == VIEW]
for metric in ("pr", "cka"):
    fig, axes = plt.subplots(4, 4, figsize=(17, 13), constrained_layout=True)
    for ax, (name, rows) in zip(axes.flat, selected.groupby("program", sort=False)):
        if metric == "pr":
            ax.plot(rows.step, rows.actual_pr, "o-", ms=3, label="Observed")
            ax.plot(rows.step, rows.expected_pr, "--", label="Encoded RASP values")
            ax.set_ylabel("Participation ratio")
        else:
            ax.plot(rows.step, rows.adjacent_cka, "o-", ms=3, color="gray", label="Adjacent CKA")
            writes = rows[rows.new_variable_count > 0]
            ax.scatter(writes.step, writes.adjacent_cka, color="darkorange", s=28, label="Scheduled write")
            for step in writes.step:
                ax.axvline(step, color="darkorange", alpha=0.25, lw=2)
            if rows.adjacent_cka.isna().all():
                ax.text(0.5, 0.5, "CKA undefined", ha="center", transform=ax.transAxes)
            ax.set(ylim=(-0.05, 1.05), ylabel="CKA")
        ax.set(xticks=rows.step, xticklabels=rows.stage)
        ax.set_xlim(-0.2, rows.step.max() + 0.2)
        ax.tick_params(axis="x", rotation=90, labelsize=7)
        ax.set_title(name, fontsize=9)
    for ax in list(axes.flat)[len(programs):]:
        ax.axis("off")
    axes.flat[0].legend(fontsize=7)
    fig.suptitle(f"{VIEW}: {'PR versus known values' if metric == 'pr' else 'CKA versus scheduled writes'}")
    fig.savefig(FIGURE_DIR / f"{metric}_{VIEW}.png", bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Inspect · causal predictions for A/B

For each target, choose the first donor probe at the same position with a different
final output class. Patch only that target's **A previous-token value** or **B histogram
count**, immediately after its writer. Values use valid compiler encodings; BOS and
other positions are untouched. A patches the output itself and is a direct readout
control; B tests propagation through the downstream repeated-token classifier.

First require each unmodified manual trace to agree with the assembled model,
including residuals. Then continue the manual circuit after substitution and compare
its entire output sequence and scores with downstream RASP reevaluation under the same
substitution. The unchanged-value condition should leave the output unchanged.

These are interventions in the verified manual circuit. Donors are selected to change
the expected answer, so the change rate is not an unbiased estimate of variable
importance. A patched intermediate state need not correspond to any natural input.

In [ ]:
interventions = result["interventions"]
summary = interventions.groupby(["program", "condition"]).agg(
    cases=("probe_id", "size"),
    output_matches=("output_matches", "sum"),
    scores_close=("scores_close", "sum"),
    max_abs_score_error=("max_abs_score_error", "max"),
)
display(summary)
display(interventions.groupby(["program", "condition"], sort=False).head(1)[[
    "program", "condition", "tokens", "position", "original_value", "patched_value",
    "baseline_output", "expected_output", "actual_output", "changed_positions",
]])
failures = interventions[~interventions.output_matches | ~interventions.scores_close]
print(f"{len(failures)} intervention/control comparisons disagree with predictions.")
if not failures.empty:
    display(failures)

## Interpret and locate results

Repeated targets retain their weight and are not independent trials. Interpret the
tables above for this execution; changing probes or settings may change the findings.

Each run exports `stages.csv`, `lanes.csv`, `variables.csv`, `values.csv`,
`interventions.csv`, the per-program probes, and `manifest.json`. Variable IDs join
the schedules, expected values, and lane comparisons. The numerical manifest's
completion status excludes the subsequent notebook plotting stage.


In [ ]:
print(f"Figures: {len(list(FIGURE_DIR.glob('*.png')))}")
print(f"Tables and manifest: {RUN_DIR}")

**Next:** [05 · Compare ground-truth agreement across P01–P12](05_comparison.ipynb).
